# DFU Repair-7 v1.7 CPU ONLY
CPU-only repair path for users without Colab GPU quota. Preserves the 38 good trials and trains only the exact seven incompatible fold-1 trials.


In [ ]:
# DFU Repair-7 v1.7 CPU ONLY
# CPU repair for the exact seven incompatible fold-1 trials.
# The immutable direct v1.6 notebook supplies recovery logic and embedded V4 evidence.

import json, re, urllib.request

DIRECT_V16_URL = "https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/b0f7d6953ab59f02d9ec36ff5784bbf6145c8da7/notebooks/DFU_Repair7_Only_Preserve38_v1_6_DIRECT_Colab.ipynb"

raw = urllib.request.urlopen(DIRECT_V16_URL, timeout=120).read()
nb = json.loads(raw.decode("utf-8"))
code_cells = [c for c in nb.get("cells", []) if c.get("cell_type") == "code"]
if len(code_cells) != 1:
    raise RuntimeError(f"Expected exactly one direct v1.6 code cell, found {len(code_cells)}")
script = "".join(code_cells[0]["source"])

required = [
    "DFU REPAIR-7 v1.6 ONLY",
    "only 7 incompatible fold-1 trials may train",
    "0b931520810c3aa5c3c5bcc7abff91f2c8b0c1e6aa1a739cdf2ebef3ae0886a9",
]
missing = [m for m in required if m not in script]
if missing:
    raise RuntimeError(f"Unexpected direct v1.6 content; missing markers: {missing}")

# Remove notebook-level GPU-only gate.
gpu_gate = re.compile(
    r'import torch\n'
    r'if not torch\.cuda\.is_available\(\):\n'
    r'\s+raise RuntimeError\("GPU runtime required\.[^\n]*"\)\n'
    r'print\("GPU:", torch\.cuda\.get_device_name\(0\)\)'
)
script, n_gate = gpu_gate.subn(
    'import torch\nprint("Compute device: CPU (forced repair mode)")',
    script,
    count=1,
)
if n_gate != 1:
    raise RuntimeError(f"CPU patch could not replace GPU gate exactly once; count={n_gate}")

# After reliable_runner_v2 is imported, replace only train_trial with a CPU-safe copy.
anchor = "\nrr.atomic_torch"
pos = script.find(anchor)
if pos < 0:
    raise RuntimeError("CPU patch anchor rr.atomic_torch not found; refusing execution")

cpu_patch = r'''
# ---- v1.7 CPU-safe training patch ----
import inspect as _repair_inspect
_cpu_train_src = _repair_inspect.getsource(rr.train_trial)
_replacements = [
    (
        '    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n'
        '    if device.type != "cuda":\n'
        '        raise RuntimeError("GPU runtime is required; training was not started.")\n',
        '    device = torch.device("cpu")\n'
    ),
    (
        '    scaler = torch.amp.GradScaler("cuda", enabled=cfg.USE_AMP)\n',
        '    scaler = torch.amp.GradScaler("cpu", enabled=False)\n'
    ),
    (
        '            with torch.amp.autocast("cuda", enabled=cfg.USE_AMP):\n',
        '            with torch.amp.autocast("cpu", enabled=False):\n'
    ),
]
for _old, _new in _replacements:
    _count = _cpu_train_src.count(_old)
    if _count != 1:
        raise RuntimeError(f"CPU train patch expected one source match, found {_count}: {_old[:70]!r}")
    _cpu_train_src = _cpu_train_src.replace(_old, _new, 1)
exec(compile(_cpu_train_src, "reliable_runner_v2_cpu_train_trial.py", "exec"), rr.__dict__)
print("CPU-safe train_trial patch: PASS")
# ---- end CPU patch ----
'''
script = script[:pos] + "\n" + cpu_patch + script[pos:]

# Disable AMP and reduce CPU memory pressure after config build.
cfg_anchor = "cfg = rr.build_config(settings)\n"
if script.count(cfg_anchor) != 1:
    raise RuntimeError(f"CPU config anchor count != 1: {script.count(cfg_anchor)}")
script = script.replace(
    cfg_anchor,
    cfg_anchor
    + 'cfg.USE_AMP = False\n'
    + 'cfg.BATCH_SIZE = 4\n'
    + 'cfg.NUM_WORKERS = min(2, max(0, (os.cpu_count() or 1) - 1))\n'
    + 'settings.batch_size = 4\n'
    + 'settings.num_workers = cfg.NUM_WORKERS\n'
    + 'torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))\n'
    + 'print(f"CPU training config: batch={cfg.BATCH_SIZE}, workers={cfg.NUM_WORKERS}, AMP={cfg.USE_AMP}, threads={torch.get_num_threads()}")\n',
    1,
)

script = script.replace("DFU REPAIR-7 v1.6 ONLY", "DFU REPAIR-7 v1.7 CPU ONLY", 1)
script = script.replace('"repair_version": "v1.6"', '"repair_version": "v1.7-cpu"', 1)

# Static safety assertions before execution.
if script.count("rr.train_trial(") != 1:
    raise RuntimeError(f"Safety check: expected exactly one rr.train_trial call, found {script.count('rr.train_trial(')}")
for forbidden in ["rr.make_outer_folds(", "rr.assign_duplicate_groups(", "rr.build_manifest("]:
    if forbidden in script:
        raise RuntimeError(f"Safety check: forbidden split-regeneration call found: {forbidden}")

print("Pinned Repair-7 v1.7 CPU patch verification: PASS")
print("GPU requirement: REMOVED")
print("Training device: CPU ONLY")
print("38 good trials: READ-ONLY")
print("Authorized retraining: EXACT 7 fold-1 repair identities")
exec(compile(script, "DFU_Repair7_v1_7_CPU.py", "exec"), globals())
